In [ ]:
# End-to-End ML Pipeline with Feature Engineering and Cross Validation
import polars as pl
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

np.random.seed(42)

df = pl.DataFrame({
    "StudyHours": np.random.randint(1, 10, 200),
    "SleepHours": np.random.randint(4, 9, 200),
    "Attendance": np.random.randint(50, 100, 200),
    "PreviousMarks": np.random.randint(30, 95, 200)
})

df = df.with_columns([
    (pl.col("StudyHours") + pl.col("Attendance")).alias("Engagement"),
    ((pl.col("StudyHours") * 0.3 +
      pl.col("Attendance") * 0.3 +
      pl.col("PreviousMarks") * 0.4) > 65)
    .cast(pl.Int64)
    .alias("Pass")
])

X = df.select(["StudyHours","SleepHours","Attendance","PreviousMarks","Engagement"]).to_numpy()
y = df.select("Pass").to_numpy().ravel()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

model1 = LogisticRegression()
model1.fit(X_train, y_train)

model2 = RandomForestClassifier(n_estimators=100)
model2.fit(X_train, y_train)

pred1 = model1.predict(X_test)
pred2 = model2.predict(X_test)

print("Logistic Accuracy:", accuracy_score(y_test, pred1))
print("Random Forest Accuracy:", accuracy_score(y_test, pred2))

cv_scores = cross_val_score(model2, X_train, y_train, cv=5)
print("Cross Validation:", cv_scores.mean())